# Phase 3: GNN Risk Prediction Model Training

This notebook trains a Graph Neural Network to predict bank risk levels.

In [2]:
# Cell 1: Setup and Imports
import sys
import os

os.chdir('/Users/aryan/datathon/resili-net')
sys.path.insert(0, '/Users/aryan/datathon/resili-net')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

from src.data_loader import fetch_and_cache_market_data, engineer_features, get_market_data_for_timestep
from src.graph_builder import build_graph_for_timestep
from src.models import BankRiskGNN
from src.config import BANK_TICKERS
from torch_geometric.data import Data

# Define date range
START_DATE = '2005-01-01'
END_DATE = '2024-12-31'

torch.manual_seed(42)
np.random.seed(42)

print("✅ All imports successful!")
print(f"Date range: {START_DATE} to {END_DATE}")

✅ All imports successful!
Date range: 2005-01-01 to 2024-12-31


In [3]:
# Cell 2: Load Data
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

historical_data = fetch_and_cache_market_data(START_DATE, END_DATE)
processed_data = engineer_features(historical_data)

print(f"\n📊 Data loaded: {len(processed_data)} rows")
print(f"📊 Date range: {processed_data.index.min()} to {processed_data.index.max()}")

LOADING DATA
Cache not found. Fetching historical data for 11 tickers...


[*********************100%***********************]  11 of 11 completed


Historical data cached successfully at data/historical_market_data_INDIA.pkl
Starting feature engineering...
Feature engineering complete.

📊 Data loaded: 7549 rows
📊 Date range: 2008-02-05 00:00:00 to 2024-11-14 00:00:00


In [4]:
# Cell 3: Build Graph Snapshots
print("=" * 60)
print("BUILDING GRAPH SNAPSHOTS")
print("=" * 60)

unique_dates = processed_data.index.unique().sort_values()
print(f"\n📊 Total unique dates: {len(unique_dates)}")

volatility_threshold = processed_data[processed_data['Ticker'].isin(BANK_TICKERS)]['realized_volatility'].quantile(0.75)
print(f"📊 Risk threshold (75th percentile volatility): {volatility_threshold:.4f}")

feature_cols = ['realized_volatility', 'price_momentum', 'beta_market', 'shadow_risk_gap', 'implied_volatility']

data_pairs = []
skipped = 0

for date in unique_dates:
    day_data = get_market_data_for_timestep(processed_data, date)
    if not day_data or len(day_data) < 3:
        skipped += 1
        continue
    
    G = build_graph_for_timestep(date, processed_data, day_data)
    if len(G.nodes()) < 2:
        skipped += 1
        continue
    
    node_features = []
    node_labels = []
    node_names = []
    
    for node in G.nodes():
        node_data = G.nodes[node]
        if node_data.get('type') != 'bank':
            continue
        
        features = []
        for col in feature_cols:
            val = node_data.get(col, 0.0)
            if pd.isna(val):
                val = 0.0
            features.append(val)
        
        node_features.append(features)
        node_names.append(node)
        
        vol = node_data.get('realized_volatility', 0.0)
        label = 1.0 if vol > volatility_threshold else 0.0
        node_labels.append(label)
    
    if len(node_features) < 2:
        skipped += 1
        continue
    
    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor(node_labels, dtype=torch.float).view(-1, 1)
    
    node_to_idx = {node: i for i, node in enumerate(node_names)}
    edge_list = []
    edge_weights = []
    
    for u, v, data in G.edges(data=True):
        if u in node_to_idx and v in node_to_idx:
            edge_list.append([node_to_idx[u], node_to_idx[v]])
            edge_list.append([node_to_idx[v], node_to_idx[u]])
            weight = abs(data.get('weight', 0.5))
            edge_weights.extend([weight, weight])
    
    if len(edge_list) == 0:
        for i in range(len(node_names)):
            edge_list.append([i, i])
            edge_weights.append(1.0)
    
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_weights, dtype=torch.float).view(-1, 1)
    
    pyg_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data_pairs.append((pyg_data, y))

print(f"\n📊 Created {len(data_pairs)} graph snapshots")

all_labels = []
for graph, labels in data_pairs:
    all_labels.extend(labels.numpy().flatten().tolist())
all_labels = np.array(all_labels)

n_high_risk = (all_labels == 1).sum()
n_low_risk = (all_labels == 0).sum()
print(f"\n📊 Class Distribution:")
print(f"   Low Risk: {n_low_risk} ({100*n_low_risk/len(all_labels):.1f}%)")
print(f"   High Risk: {n_high_risk} ({100*n_high_risk/len(all_labels):.1f}%)")

BUILDING GRAPH SNAPSHOTS

📊 Total unique dates: 830
📊 Risk threshold (75th percentile volatility): 0.3711

📊 Created 830 graph snapshots

📊 Class Distribution:
   Low Risk: 3735 (75.0%)
   High Risk: 1245 (25.0%)


In [5]:
# Cell 4: Data Splitting
print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

n_samples = len(data_pairs)
n_train = int(0.7 * n_samples)
n_val = int(0.15 * n_samples)

indices = np.random.permutation(n_samples)
train_data = [data_pairs[i] for i in indices[:n_train]]
val_data = [data_pairs[i] for i in indices[n_train:n_train + n_val]]
test_data = [data_pairs[i] for i in indices[n_train + n_val:]]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

all_features = torch.cat([g[0].x for g in train_data], dim=0)
feat_mean = all_features.mean(dim=0, keepdim=True)
feat_std = all_features.std(dim=0, keepdim=True) + 1e-8

train_labels = np.concatenate([g[1].numpy().flatten() for g in train_data])
n_pos = (train_labels == 1).sum()
n_neg = (train_labels == 0).sum()
pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
print(f"pos_weight: {pos_weight:.2f}")

DATA PREPROCESSING
Train: 581, Val: 124, Test: 125
pos_weight: 2.86


In [6]:
# Cell 5: Model Setup
INPUT_DIM = train_data[0][0].x.shape[1]
HIDDEN_DIM = 128
NUM_HEADS = 8
DROPOUT = 0.3
LR = 0.0005
NUM_EPOCHS = 150
PATIENCE = 25

model = BankRiskGNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=1, num_heads=NUM_HEADS, dropout=DROPOUT)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 1,226,625


In [7]:
# Cell 6: Training
print("=" * 60)
print("TRAINING")
print("=" * 60)

train_losses, val_losses, val_aucs = [], [], []
best_val_auc = 0.0
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_samples = 0
    
    for graph, labels in train_data:
        optimizer.zero_grad()
        g = graph.clone()
        g.x = (g.x - feat_mean) / feat_std
        g.x = torch.nan_to_num(g.x, nan=0.0)
        
        outputs = model(g.x, g.edge_index, g.edge_attr)
        loss = criterion(outputs, labels.view(-1, 1).float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item() * len(labels)
        n_samples += len(labels)
    
    train_losses.append(epoch_loss / n_samples)
    
    # Validation
    model.eval()
    val_probs, val_labels_list = [], []
    val_loss = 0.0
    n_val = 0
    
    with torch.no_grad():
        for graph, labels in val_data:
            g = graph.clone()
            g.x = (g.x - feat_mean) / feat_std
            g.x = torch.nan_to_num(g.x, nan=0.0)
            
            outputs = model(g.x, g.edge_index, g.edge_attr)
            loss = criterion(outputs, labels.view(-1, 1).float())
            val_loss += loss.item() * len(labels)
            n_val += len(labels)
            
            probs = torch.sigmoid(outputs).squeeze().numpy()
            if probs.ndim == 0:
                val_probs.append(probs.item())
            else:
                val_probs.extend(probs.tolist())
            val_labels_list.extend(labels.numpy().flatten().tolist())
    
    val_losses.append(val_loss / n_val if n_val > 0 else 0)
    
    try:
        val_auc = roc_auc_score(val_labels_list, val_probs) if len(np.unique(val_labels_list)) > 1 else 0.5
    except:
        val_auc = 0.5
    val_aucs.append(val_auc)
    scheduler.step(val_auc)
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_no_improve = 0
        os.makedirs('models', exist_ok=True)
        torch.save({'model_state_dict': model.state_dict(), 'val_auc': best_val_auc, 'feat_mean': feat_mean, 'feat_std': feat_std}, 'models/gnn_risk_model.pt')
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train loss: {train_losses[-1]:.4f} | Val loss: {val_losses[-1]:.4f} | AUC: {val_auc:.4f}")
    
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break

print(f"\n✅ Best AUC: {best_val_auc:.4f}")

TRAINING
Epoch  10 | Train: 0.3926 | Val: 0.1698 | AUC: 0.9896
Epoch  20 | Train: 0.3292 | Val: 0.1207 | AUC: 0.9933
Epoch  30 | Train: 0.3739 | Val: 0.1645 | AUC: 0.9917
Epoch  40 | Train: 0.3285 | Val: 0.1134 | AUC: 0.9941
Epoch  50 | Train: 0.3190 | Val: 0.1492 | AUC: 0.9932
Epoch  60 | Train: 0.3204 | Val: 0.1479 | AUC: 0.9921

Early stopping at epoch 61

✅ Best AUC: 0.9943


In [ ]:
# Cell 7: Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(val_aucs, label='Val AUC')
axes[1].axhline(y=0.5, color='r', linestyle='--')
axes[1].set_title('AUC-ROC')
axes[1].legend()
plt.tight_layout()
plt.savefig('models/training_curves.png')
plt.show()

In [ ]:
# Cell 8: Evaluation
print("=" * 60)
print("EVALUATION")
print("=" * 60)

checkpoint = torch.load('models/gnn_risk_model.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model with AUC: {checkpoint['val_auc']:.4f}")

model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for graph, labels in test_data:
        g = graph.clone()
        g.x = (g.x - feat_mean) / feat_std
        g.x = torch.nan_to_num(g.x, nan=0.0)
        
        outputs = model(g.x, g.edge_index, g.edge_attr)
        probs = torch.sigmoid(outputs).squeeze().numpy()
        if probs.ndim == 0:
            all_probs.append(probs.item())
        else:
            all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().flatten().tolist())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print(f"\nProb range: [{all_probs.min():.4f}, {all_probs.max():.4f}], Mean: {all_probs.mean():.4f}")

try:
    test_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.5
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    optimal_threshold = thresholds[np.argmax(tpr - fpr)]
except:
    test_auc, optimal_threshold = 0.5, 0.5

print(f"Test AUC: {test_auc:.4f}")
print(f"Optimal threshold: {optimal_threshold:.4f}")

all_preds = (all_probs >= optimal_threshold).astype(float)
print(f"\nPredicted High Risk: {int(all_preds.sum())} / Actual: {int(all_labels.sum())}")
print("\n" + classification_report(all_labels, all_preds, target_names=['Low Risk', 'High Risk'], zero_division=0))
print(confusion_matrix(all_labels, all_preds))

In [ ]:
# Cell 9: Sample Predictions
print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

if len(test_data) > 0:
    sample_graph, _ = test_data[0]
    g = sample_graph.clone()
    g.x = (g.x - feat_mean) / feat_std
    g.x = torch.nan_to_num(g.x, nan=0.0)
    
    with torch.no_grad():
        probs = torch.sigmoid(model(g.x, g.edge_index, g.edge_attr)).squeeze().numpy()
    
    for i, bank in enumerate(BANK_TICKERS[:len(probs) if probs.ndim > 0 else 1]):
        p = probs[i] if probs.ndim > 0 else probs.item()
        risk = '🔴 HIGH' if p >= optimal_threshold else '🟢 LOW'
        print(f"{bank:<20} {p:.4f}  {risk}")

# Save config
with open('models/model_config.json', 'w') as f:
    json.dump({'optimal_threshold': float(optimal_threshold), 'test_auc': float(test_auc), 'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM}, f, indent=2)

print("\n✅ Phase 3 Complete!")